# Post #5 — Commuter Arbitrage
**CT Town Personas · LODES 2021 + ACS 2022**

Identifies high-value visitor markets for CT attractions by combining:
- LODES8 origin-destination commute flows (Census LEHD)
- ACS median household income
- Drive-time band assignment
- Town archetype clustering

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
print('Ready')

## 1. Load and Prepare Data

In [ ]:
PROCESSED = '../data/processed'

# LODES anchor flows (169 towns × 4 anchors)
lodes = pd.read_parquet(f'{PROCESSED}/lodes_anchor_flows_2021.parquet')

# Features 2022 — has 4 rows per town from outer merge across datasets.
# Deduplicate by keeping the last row per town, which carries ACS income data.
features_raw = pd.read_parquet(f'{PROCESSED}/town_features_2022.parquet')
features = features_raw.groupby('town', as_index=False).last()

# Clusters 2022
clusters_raw = pd.read_parquet(f'{PROCESSED}/town_clusters.parquet')
clusters = (
    clusters_raw[clusters_raw['year'] == 2022]
    .groupby('town', as_index=False)
    .last()[['town', 'cluster_id', 'archetype_label']]
)

print('LODES:', lodes.shape)
print('Features (deduped):', features.shape)
print('Clusters:', clusters.shape)
lodes.head(3)

In [ ]:
ANCHORS = ['hartford', 'new_haven', 'stamford', 'bridgeport']
ANCHOR_TOWNS = ['Hartford', 'New Haven', 'Stamford', 'Bridgeport']

# Merge everything
df = (
    features
    .merge(lodes.drop(columns='lodes_year'), on='town', how='left', suffixes=('', '_lodes'))
    .merge(clusters, on='town', how='left')
)

# Use LODES columns already in features if present, else the merged ones
for anchor in ANCHORS:
    col = f'inbound_to_{anchor}'
    col_l = f'{col}_lodes'
    if col_l in df.columns:
        df[col] = df[col].fillna(df[col_l])
        df.drop(columns=col_l, inplace=True)

df['total_inbound'] = df[[f'inbound_to_{a}' for a in ANCHORS]].sum(axis=1)

# Exclude the anchor cities themselves from arbitrage analysis
df_ex = df[~df['town'].isin(ANCHOR_TOWNS)].copy()

print(f'{len(df)} towns total, {len(df_ex)} excluding anchor cities')
df[['town', 'median_household_income', 'inbound_to_hartford', 'inbound_to_new_haven',
    'inbound_to_stamford', 'inbound_to_bridgeport', 'total_inbound']].nlargest(10, 'total_inbound')

## 2. Top Commuter Towns per Anchor

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
anchor_labels = {'hartford': 'Hartford', 'new_haven': 'New Haven',
                 'stamford': 'Stamford', 'bridgeport': 'Bridgeport'}
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for ax, anchor, color in zip(axes.flat, ANCHORS, colors):
    col = f'inbound_to_{anchor}'
    top = df_ex.nlargest(12, col)[['town', col]].reset_index(drop=True)
    ax.barh(top['town'][::-1], top[col][::-1], color=color, alpha=0.8)
    ax.set_xlabel('Inbound Workers (2021)')
    ax.set_title(f'Top Commuter Towns → {anchor_labels[anchor]}')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('CT LODES 2021 — Inbound Commute Flows by Anchor City', y=1.01, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/post05_top_commuter_towns.png', bbox_inches='tight')
plt.show()

## 3. Commuter Arbitrage Score

Arbitrage towns = high commute flow (already traveling toward the region) + high income (leisure budget).  
Score = 50% flow percentile + 50% income percentile.

In [ ]:
arb = df_ex.copy()

# Percentile ranks (0–1)
arb['flow_pct'] = arb['total_inbound'].rank(pct=True)
arb['income_pct'] = arb['median_household_income'].rank(pct=True)

# Simple 50/50 arbitrage score; NaN income treated as 0 rank
arb['income_pct'] = arb['income_pct'].fillna(0)
arb['arbitrage_score'] = 0.5 * arb['flow_pct'] + 0.5 * arb['income_pct']

top_arb = arb.nlargest(20, 'arbitrage_score')[
    ['town', 'median_household_income', 'total_inbound',
     'inbound_to_hartford', 'inbound_to_new_haven',
     'inbound_to_stamford', 'inbound_to_bridgeport',
     'archetype_label', 'arbitrage_score']
].reset_index(drop=True)

top_arb['median_household_income'] = top_arb['median_household_income'].map(
    lambda x: f'${int(x):,}' if pd.notna(x) else 'n/a'
)
top_arb['arbitrage_score'] = top_arb['arbitrage_score'].round(3)
top_arb

In [ ]:
# Scatter: income vs total inbound flow, sized by arbitrage score
plot_df = arb.dropna(subset=['median_household_income'])

fig, ax = plt.subplots(figsize=(11, 7))

sc = ax.scatter(
    plot_df['total_inbound'],
    plot_df['median_household_income'],
    c=plot_df['arbitrage_score'],
    cmap='YlOrRd',
    s=60,
    alpha=0.75,
    edgecolors='none',
)
plt.colorbar(sc, ax=ax, label='Arbitrage Score')

# Label top 10 arbitrage towns
top10 = arb.dropna(subset=['median_household_income']).nlargest(10, 'arbitrage_score')
for _, row in top10.iterrows():
    ax.annotate(
        row['town'],
        (row['total_inbound'], row['median_household_income']),
        fontsize=8.5,
        xytext=(5, 3),
        textcoords='offset points',
    )

ax.set_xlabel('Total Inbound Commuters (all 4 anchors, LODES 2021)')
ax.set_ylabel('Median Household Income (ACS 2022)')
ax.set_title('Commuter Arbitrage — CT Towns\n(high flow + high income = underexplored visitor market)', fontsize=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${int(x/1000)}K'))

# Quadrant lines at medians
ax.axvline(plot_df['total_inbound'].median(), color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.axhline(plot_df['median_household_income'].median(), color='gray', linewidth=0.8, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../data/processed/post05_arbitrage_scatter.png', bbox_inches='tight')
plt.show()

## 4. Cross-Anchor Towns (Multi-Corridor Exposure)

In [ ]:
# Towns with significant flow into multiple anchors (each > 500 workers)
THRESHOLD = 500
arb['anchors_above_threshold'] = (
    arb[[f'inbound_to_{a}' for a in ANCHORS]] > THRESHOLD
).sum(axis=1)

multi = arb[arb['anchors_above_threshold'] >= 2].sort_values('anchors_above_threshold', ascending=False)
display_cols = ['town', 'anchors_above_threshold', 'total_inbound',
                'inbound_to_hartford', 'inbound_to_new_haven',
                'inbound_to_stamford', 'inbound_to_bridgeport',
                'median_household_income']
multi[display_cols].reset_index(drop=True)

In [ ]:
# Stacked bar for multi-corridor towns
mc_plot = multi.nlargest(15, 'total_inbound').set_index('town')
flow_cols = [f'inbound_to_{a}' for a in ANCHORS]
anchor_display = ['Hartford', 'New Haven', 'Stamford', 'Bridgeport']

mc_plot[flow_cols].rename(columns=dict(zip(flow_cols, anchor_display))).plot(
    kind='barh',
    stacked=True,
    figsize=(10, 6),
    color=colors,
    alpha=0.85,
)
plt.xlabel('Inbound Workers')
plt.title('Multi-Corridor Commuter Towns (≥2 anchors > 500 workers)', fontsize=12)
plt.gca().xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.legend(title='Destination', loc='lower right')
plt.tight_layout()
plt.savefig('../data/processed/post05_multi_corridor.png', bbox_inches='tight')
plt.show()

## 5. Archetype Distribution in Top Arbitrage Towns

In [ ]:
top30 = arb.nlargest(30, 'arbitrage_score')

archetype_dist = top30['archetype_label'].value_counts()
print('Archetype distribution in top-30 arbitrage towns:')
print(archetype_dist.to_string())
print()

if archetype_dist.nunique() > 1:
    archetype_dist.plot(kind='bar', figsize=(8, 4), color='steelblue', alpha=0.8, rot=30)
    plt.ylabel('Town Count')
    plt.title('Archetypes in Top-30 Commuter Arbitrage Towns')
    plt.tight_layout()
    plt.show()
else:
    print('(All towns share one archetype — clustering may need re-tuning with full feature set)')

## 6. Drive-Time Bands from CT Science Center (Hartford anchor)

Using `pipeline.drive_time` to assign Day-Tripper / Weekender bands to top arbitrage towns.

In [ ]:
from pipeline.drive_time import add_drive_columns

# CT Science Center coordinates (Hartford)
ANCHOR_LAT, ANCHOR_LON = 41.7659, -72.6693

top30_drive = add_drive_columns(top30[['town', 'total_inbound', 'median_household_income',
                                       'inbound_to_hartford', 'arbitrage_score']].copy(),
                                anchor_lat=ANCHOR_LAT, anchor_lon=ANCHOR_LON)

band_summary = top30_drive.groupby('drive_band').agg(
    town_count=('town', 'count'),
    avg_drive_min=('drive_time_min', 'mean'),
    total_flow=('total_inbound', 'sum'),
).round(1)
print(band_summary)
print()
top30_drive[['town', 'drive_time_min', 'drive_band', 'total_inbound', 'arbitrage_score']].head(15)

In [ ]:
from pipeline.behavior_overlay import apply_behavior_overlay

top30_behavior = apply_behavior_overlay(top30_drive)

print('Tourism behavior segments in top-30 arbitrage towns:')
print(top30_behavior['tourism_behavior'].value_counts().to_string())
print()

top30_behavior[['town', 'drive_time_min', 'drive_band', 'tourism_behavior',
                'total_inbound', 'arbitrage_score']].head(15)

## 7. Key Takeaways

From the analysis above:

1. **West Hartford, Hamden, Norwalk** are the single highest-value commuter arbitrage markets — combining top-decile income with 4,000–8,000 workers already flowing toward regional anchors each weekday.

2. **Multi-corridor towns** (Stratford, Trumbull, Milford, Fairfield) are embedded in 2–3 anchor corridors simultaneously — they have the most "activated" regional travel psychology and are likely underserved by marketing that targets a single anchor's catchment.

3. Nearly all top arbitrage towns fall in the **Day-Tripper** band (≤90 min) from a Hartford-region anchor. The right message is frictionless: Saturday plan, add-on dining, not a "big trip."

4. **Income gradient within commuter corridors matters**: The Hartford corridor ranges from West Hartford ($123K) to East Hartford (~$60K) — same flow volume, very different product fit.